## Configuration

### Model config
Load config

In [92]:
import yaml
import torch
import wandb
import pandas as pd
import torch.nn as nn

# yaml_path = 'configs/default.yaml'
yaml_path = 'configs/best.yaml'
with open(yaml_path, "r") as f:
    config = yaml.safe_load(f)

wandb_path = 'configs/wandb.yaml'
with open(wandb_path, "r") as f:
    wandb_config = yaml.safe_load(f)

Flexible model class

In [93]:
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_layers, output_dim,
                 dropout=0.0, batch_norm=False, activation="relu"):
        super().__init__()
        layers = []
        act_fn = {
            "relu": nn.ReLU(),
            "tanh": nn.Tanh(),
            "sigmoid": nn.Sigmoid()
        }[activation]
        
        for h in hidden_layers:
            layers.append(nn.Linear(input_dim, h))
            if batch_norm:
                layers.append(nn.BatchNorm1d(h))
            layers.append(act_fn)
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            input_dim = h

        layers.append(nn.Linear(input_dim, output_dim))
        layers.append(nn.Sigmoid())

        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


Split data and prepare everything

In [94]:
from sklearn.model_selection import train_test_split

model_config = config['model']
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load CSVs
df = pd.read_csv("data/bioresponse_filtered.csv")

X = df.drop('target', axis=1)
y = df['target']

# Base train/test selection
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.10, random_state=42, stratify=y
)

# Validation subset from train set
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.1111,  # 0.1111 * 0.9 = 0.10 (10%)
    random_state=42, stratify=y_train_full
)

total = len(df)

# Print shapes + percentages
print(f"Training set shape: {X_train.shape} ({len(X_train)/total:.1%})")
print(f"Test set shape: {X_test.shape} ({len(X_test)/total:.1%})")
print(f"Validation set shape: {X_val.shape} ({len(X_val)/total:.1%})")

model = MLP(
    input_dim=X_train.shape[1],
    hidden_layers=model_config['hidden_layers'],
    output_dim=model_config['output_dim'],
    dropout=model_config['dropout'],
    batch_norm=model_config['batch_norm'],
    activation=model_config['activation']
).to(device)

optimizer_dict = {
    "adam": torch.optim.Adam,
    "sgd": torch.optim.SGD,
    "rmsprop": torch.optim.RMSprop,
    "adamw": torch.optim.AdamW,
}
optimizer_class = optimizer_dict[config['training']['optimizer']]

optimizer = optimizer_class(
    model.parameters(),
    lr=float(config['training']['learning_rate']),
)

loss_fn     = nn.BCELoss()
X_train_t   = torch.tensor(X_train.values, dtype=torch.float32).to(device)
y_train_t   = torch.tensor(y_train.values, dtype=torch.float32).to(device)
X_test_t    = torch.tensor(X_test.values, dtype=torch.float32).to(device)
y_test_t    = torch.tensor(y_test.values, dtype=torch.float32).to(device)
X_val_t     = torch.tensor(X_val.values, dtype=torch.float32).to(device)
y_val_t     = torch.tensor(y_val.values, dtype=torch.float32).to(device)

Training set shape: (3000, 33) (80.0%)
Test set shape: (376, 33) (10.0%)
Validation set shape: (375, 33) (10.0%)


### Wandb

Wandb login tutorial:

1. `pip install wandb`
2. 
```python
import wandb 
wandb.login()
```
3. you are prompted API key (which you find on wandb.ai website after you log in)

In [95]:
wandb.login()

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


True

In [96]:
wandb.login()
wandb.init(
    project=wandb_config["wandb"]["project"],
    entity=wandb_config["wandb"]["entity"],
    name=wandb_config["wandb"].get("run_name", wandb_config["wandb"]["name"]),
    group=config["experiment"].get("group", None),
    notes=wandb_config["wandb"].get("notes", ""),
    config=wandb_config,
)
wandb.watch(model, log="all")


wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


epoch,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
train_loss,█▇▇▆▆▇▇▆▇▇▇▆▆▆▄▅▅▄▄▄▅▄▄▄▃▃▂▂▄▃▂▁▂▁▁▂▁▂▁▂
val_accuracy,▂▃▄▄▃▁▄▄▄▄▄▄▄▄▄▄▅▃▄▄▅▆▄▄▄▅▄▄▅▆▆▆▂▇▃▄▄▄▃█
val_f1,▆▆███▁█████████▇▆▇██████▃▄██▇███▂▇▃▆██▂█
val_loss,▆▃▃▃▃▃▃▃▃▃▃▃▃▅▃▂▃█▇▃▂▂▂▄▂▃▇█▅▃▂▂▅▂▄▃▂▄▅▁
epoch,40
train_loss,0.65005
val_accuracy,0.64267
val_f1,0.69545
val_loss,0.63478


## Training

Load datasets with Dataloaders

In [97]:
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, f1_score

train_dataset = TensorDataset(X_train_t, y_train_t)
test_dataset = TensorDataset(X_test_t, y_test_t)
val_dataset = TensorDataset(X_val_t, y_val_t)

train_loader = DataLoader(
    train_dataset, 
    batch_size=config['training']['batch_size'],
    shuffle=True
)
val_loader = DataLoader(
    val_dataset,
    batch_size=config['training']['batch_size'],
    shuffle=False
)

Actual training

In [98]:
def train():
    # re-load config (if changed during setup)
    with open(yaml_path, "r") as f:
        config = yaml.safe_load(f)
    best_val_loss = float('inf')
    patience = 10
    counter = 0
    epochs = config['training']['epochs']
    log_interval = config['experiment']['log_interval']

    for epoch in range(1, epochs + 1):
        model.train() # toggle model to training mode
        training_loss = 0.0

        ## The magic 3 steps: forward, backward, step
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()                   # reset gradients
            outputs = model(X_batch).squeeze()      # forward pass
            
            loss = loss_fn(outputs, y_batch)        # compute loss
            loss.backward()                         # backward pass

            optimizer.step()                        # update weights
            training_loss += loss.item()

        avg_training_loss = training_loss / len(train_loader)

        ## Train test phase
        model.eval()  # toggle model to evaluation mode
        val_loss = 0.0
        y_true, y_pred = [], []
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                outputs = model(X_batch).squeeze()
                loss = loss_fn(outputs, y_batch)
                val_loss += loss.item()

                y_true.extend(y_batch.cpu().numpy())
                y_pred.extend((outputs.cpu().numpy() >= 0.5).astype(int))
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            counter = 0
        else:
            counter += 1
            if counter > patience:
                print('Overfiiting of data.')
                break
        avg_val_loss = val_loss / len(val_loader)
        val_accuracy = accuracy_score(y_true, y_pred)
        val_f1 = f1_score(y_true, y_pred)

        # === LOGGING ===
        if epoch % log_interval == 0 or epoch == 1:
            print(f"Epoch {epoch}/{epochs} - "
                f"Train Loss: {avg_training_loss:.4f} - "
                f"Val Loss: {avg_val_loss:.4f} - "
                f"Val Acc: {val_accuracy:.4f}"
            )

        if wandb_config["wandb"]["enabled"]:
            wandb.log({
                "epoch": epoch,
                "train_loss": avg_training_loss,
                "val_loss": avg_val_loss,
                "val_accuracy": val_accuracy,
                "val_f1": val_f1,
            })

    if wandb_config["wandb"]["enabled"]:
        # after the loop
        wandb.log({
            "final_conf_mat": wandb.plot.confusion_matrix(
                probs=None,
                y_true=y_true,
                preds=y_pred,
                class_names=["negative", "positive"]
            )
        })


train()

Epoch 1/40 - Train Loss: 0.7140 - Val Loss: 0.7101 - Val Acc: 0.5120
Epoch 10/40 - Train Loss: 0.6893 - Val Loss: 0.6917 - Val Acc: 0.5413
Overfiiting of data.


Test the model after training

In [99]:
def evaluate():
    model.eval()
    test_loss = 0.0
    y_true, y_pred = [], []

    test_loader = DataLoader(
        test_dataset,
        batch_size=config['training']['batch_size'],
        shuffle=False
    )

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            outputs = model(X_batch).squeeze()
            loss = loss_fn(outputs, y_batch)
            test_loss += loss.item()
            y_true.extend(y_batch.cpu().numpy())
            y_pred.extend((outputs.cpu().numpy() >= 0.5).astype(int))

    avg_test_loss = test_loss / len(test_loader.dataset)
    test_accuracy = accuracy_score(y_true, y_pred)

    print(f"\nFinal Test Results — Loss: {avg_test_loss:.4f}, Accuracy: {test_accuracy:.4f}")
    wandb.log({"test_loss": avg_test_loss, "test_accuracy": test_accuracy})

evaluate()


Final Test Results — Loss: 0.0038, Accuracy: 0.5186
